In [1]:
from openai import AsyncOpenAI                                                                                                 
from agents import Agent, Runner, RunConfig, OpenAIChatCompletionsModel                                                        
                                                                                                                                
client = AsyncOpenAI(                                                                                                          
    api_key="sk-839ab4b91b25494d910a04a8812d8cf6",                                                                             
    base_url="https://api.deepseek.com",                                                                                       
)                                                                                                                              
                                                                                                                                
model = OpenAIChatCompletionsModel(                                                                                            
    model="deepseek-v4-pro",                                                                                                   
    openai_client=client,                                                                                                      
)                                                                                                                              
                                                                                                                                
      

In [ ]:
agent = Agent(                                                                                                                 
    name="Assistant",                                                                                                          
    instructions="You are a helpful assistant.",                                                                               
    model=model,                                                                                                               
)                                                                                                                              
                                                                                                                                
result = await Runner.run(agent, "hi")    

print(result.final_output)                                                             
# print(result.final_output)  

OPENAI_API_KEY is not set, skipping trace export


Hello! How can I assist you today?


OPENAI_API_KEY is not set, skipping trace export


In [ ]:
#工具调用
import asyncio
from agents import Agent, Runner, function_tool, RunResult


@function_tool
def history_fun_fact() -> str:
    """Return a short history fact."""
    return "Sharks are older than trees."


agent = Agent(
    name="History Tutor",
    instructions="Answer history questions clearly. Use history_fun_fact when it helps.",
    model=model,
    tools=[history_fun_fact],
)



result = await Runner.run(
    agent,
    "Tell me something surprising about ancient life on Earth.",
)
print(result.final_output)




OPENAI_API_KEY is not set, skipping trace export


Here's something truly mind-boggling: **Sharks are older than trees.**

Sharks first appeared in Earth's oceans around **400 million years ago** (during the Devonian period), while the first proper trees didn't emerge until about **350 million years ago**. That means sharks predate trees by roughly 50 million years — they were roaming the seas long before forests covered the land.

To put it another way: sharks have survived *four* of Earth's five major mass extinction events, including the one that wiped out the dinosaurs. Trees, by comparison, are relative newcomers! 🌊🦈 > 🌲


OPENAI_API_KEY is not set, skipping trace export


In [11]:
print(result)

RunResult:
- Last agent: Agent(name="History Tutor", ...)
- Final output (str):
    Here's a truly mind-bending one:
    
    **Sharks are older than trees.** 🦈🌳
    
    Sharks first appeared in Earth's oceans roughly **400 million years ago** during the Devonian period. Trees, on the other hand, didn't show up until about **385 million years ago** (the first true tree being *Archaeopteris*). That means sharks were cruising the seas for around **15 million years** before the first forest ever took root on land.
    
    To put that in perspective: sharks have survived *five* mass extinctions — including the one that wiped out the dinosaurs — and yet the first trees are long extinct. They're among the most ancient and resilient life forms the planet has ever known.
- 5 new item(s)
- 2 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


In [3]:
for item in result.new_items: 
    print(item)
    print("\n"*10)

# result.new_items[4]

ReasoningItem(agent=Agent(name='History Tutor', handoff_description=None, tools=[FunctionTool(name='history_fun_fact', description='Return a short history fact.', params_json_schema={'properties': {}, 'title': 'history_fun_fact_args', 'type': 'object', 'additionalProperties': False, 'required': []}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x7f8478d90dd0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None)], mcp_servers=[], mcp_config={}, instructions='Answer history questions clearly. Use history_fun_fact when it helps.', prompt=None, handoffs=[], model=<agents.models.openai_chatcompletions.OpenAIChatCompletionsModel object at 0x7f847a16cd10>, model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, to

In [ ]:
##多智能体通信 multiagents

history_tutor_agent = Agent(
    name="History Tutor",
    model= model, 
    handoff_description="Specialist agent for historical questions",
    instructions="You answer history questions clearly and concisely.",
)

math_tutor_agent = Agent(
    name="Math Tutor",
    model = model,
    handoff_description="Specialist agent for math questions",
    instructions="You explain math step by step and include worked examples.",
)


triage_agent = Agent(
    name="Triage Agent",
    model = model,
    instructions="Route each homework question to the right specialist.",
    handoffs=[history_tutor_agent, math_tutor_agent],
)

result = await Runner.run(
        triage_agent,
        "Who was the first president of the United States?",
    )

OPENAI_API_KEY is not set, skipping trace export


OPENAI_API_KEY is not set, skipping trace export


In [5]:
for item in result.new_items: 
    print(item)
    print("\n"*3)

ReasoningItem(agent=Agent(name='Triage Agent', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='Route each homework question to the right specialist.', prompt=None, handoffs=[Agent(name='History Tutor', handoff_description='Specialist agent for historical questions', tools=[], mcp_servers=[], mcp_config={}, instructions='You answer history questions clearly and concisely.', prompt=None, handoffs=[], model=<agents.models.openai_chatcompletions.OpenAIChatCompletionsModel object at 0x7f847a16cd10>, model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, prompt_cache_retention=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None, retry=None, context_management=None), input_guardrails=[], ou

In [5]:
# #输出类型
# from pydantic import BaseModel
# from agents import Agent


# class CalendarEvent(BaseModel):
#     name: str
#     date: str
#     participants: list[str]

# agent = Agent(
#     name="Calendar extractor",
#     model= model,
#     instructions="Extract calendar events from text",
#     output_type=CalendarEvent,
# )


# result = await Runner.run(
#         agent,
#         input = "今天是几号",
#     )

In [ ]:
#管理器（agents as tools）


history_tutor_agent = Agent(
    name="History Tutor",
    model= model, 
    handoff_description="Specialist agent for historical questions",
    instructions="You answer history questions clearly and concisely.",
)

math_tutor_agent = Agent(
    name="Math Tutor",
    model = model,
    handoff_description="Specialist agent for math questions",
    instructions="You explain math step by step and include worked examples.",
)


triage_agent = Agent(
    name="Triage Agent",
    model = model,
    instructions="Route each homework question to the right specialist.",
    tools=[
        history_tutor_agent.as_tool(
            tool_name="history_expert",
            tool_description="历史知识专家.",
        ),
        math_tutor_agent.as_tool(
            tool_name="math_expert",
            tool_description="数学知识专家",
        )
    ],
)

result = await Runner.run(
        triage_agent,
        "谁是现任美国总统",
    )

OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export


OPENAI_API_KEY is not set, skipping trace export


In [7]:
for item in result.new_items: 
    print(item)
    print("\n"*3)

ReasoningItem(agent=Agent(name='Triage Agent', handoff_description=None, tools=[FunctionTool(name='history_expert', description='历史知识专家.', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x7f779649fb10>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None), FunctionTool(name='math_expert', description='数学知识专家', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': '

In [ ]:
#任务转移，多智能体


history_tutor_agent = Agent(
    name="History Tutor",
    model= model, 
    handoff_description="Specialist agent for historical questions",
    instructions="You answer history questions clearly and concisely.",
)

math_tutor_agent = Agent(
    name="Math Tutor",
    model = model,
    handoff_description="Specialist agent for math questions",
    instructions="You explain math step by step and include worked examples.",
)


triage_agent = Agent(
    name="Triage Agent",
    model=model,
    instructions="Route each homework question to the right specialist.",
    handoffs=[history_tutor_agent, math_tutor_agent],
)

result = await Runner.run(
        triage_agent,
        "谁是现任美国总统",
    )

for item in result.new_items: 
    print(item)
    print("\n"*3)

OPENAI_API_KEY is not set, skipping trace export


ReasoningItem(agent=Agent(name='Triage Agent', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='Route each homework question to the right specialist.', prompt=None, handoffs=[Agent(name='History Tutor', handoff_description='Specialist agent for historical questions', tools=[], mcp_servers=[], mcp_config={}, instructions='You answer history questions clearly and concisely.', prompt=None, handoffs=[], model=<agents.models.openai_chatcompletions.OpenAIChatCompletionsModel object at 0x7f7797770550>, model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, prompt_cache_retention=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None, retry=None, context_management=None), input_guardrails=[], ou

OPENAI_API_KEY is not set, skipping trace export


In [ ]:
from dataclasses import dataclass                                                      
from agents import Agent, Runner, RunContextWrapper, function_tool                     
                                                                                        
@dataclass                                                                             
class UserContext:                                                                     
    name: str                                                                          
    uid: str                                                                           
    is_pro_user: bool                                                                  
                                                                                        
def dynamic_instructions(                                                              
    context: RunContextWrapper[UserContext],                                           
    agent: Agent[UserContext],                                                         
) -> str:                                                                              
    user = context.context                                                             
    return f"""                                                                                             
   You are a user assistant.                                                                                   
                                                                                                               
   Current user profile:                                                                                       
   - name: {user.name}                                                                                         
   - uid: {user.uid}                                                                                                                                                    
                                                                                                               
   When the user asks about their name, id, uid, or plan,                                                      
   answer directly from this profile.                                                                          
   Do not call tools unless the requested information is missing from this profile.        
   """                             
                                                                                        
@function_tool                                                                         
def get_user_plan(context: RunContextWrapper[UserContext]) -> str:                     
    user = context.context                                                             
    return "pro" if user.is_pro_user else "free"                                       
                                                                                        
agent = Agent[UserContext](                                                            
    name="User Assistant",    
    model=model,                                                         
    instructions=dynamic_instructions,                                                 
    tools=[get_user_plan],                                                             
)                                                                                      
                                                                                        
ctx = UserContext(                                                                     
    name="Ada",                                                                        
    uid="u_001",                                                                       
    is_pro_user=True,                                                                  
)                                                                                      
                                                                                        
result = await Runner.run(                                                             
    agent,                                                                             
    "我当前等级是什么",                                                              
    context=ctx,                                                                       
)                                                                                      
                                                                                        
# print(result.final_output)  
for item in result.new_items: 
    print(item)
    print("\n"*3)  

OPENAI_API_KEY is not set, skipping trace export


ReasoningItem(agent=Agent(name='User Assistant', handoff_description=None, tools=[FunctionTool(name='get_user_plan', description='', params_json_schema={'properties': {}, 'title': 'get_user_plan_args', 'type': 'object', 'additionalProperties': False, 'required': []}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x7f7794ac1590>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None)], mcp_servers=[], mcp_config={}, instructions=<function dynamic_instructions at 0x7f7794bbf6a0>, prompt=None, handoffs=[], model=<agents.models.openai_chatcompletions.OpenAIChatCompletionsModel object at 0x7f7797770550>, model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=No

OPENAI_API_KEY is not set, skipping trace export


In [22]:
from agents.lifecycle import RunHooksBase

# 2. 业务上下文                                                                                             
@dataclass                                                                                                  
class UserContext:                                                                                          
    name: str                                                                                               
    uid: str                                                                                                
    is_pro_user: bool                                                                                       
                                                                                                            
                                                                                                            
# 3. 动态 instructions：运行时把 UserContext 注入 system prompt                                             
def dynamic_instructions(                                                                                   
    context: RunContextWrapper[UserContext],                                                                
    agent: Agent[UserContext],                                                                              
) -> str:                                                                                                   
    user = context.context                                                                                  
                                                                                                            
    return f"""                                                                                             
You are helping the current user.                                                                           
                                                                                                            
Known user profile:                                                                                         
- name: {user.name}                                                                                         
- uid: {user.uid}                                                                                           
- is_pro_user: {user.is_pro_user}                                                                           
                                                                                                            
Rules:                                                                                                      
- If asked for name or uid, answer directly from Known user profile.                                        
- Only call get_user_plan when asked about plan, subscription, tier, or membership.                         
"""                                                                                                         
                                                                                                            
                                                                                                            
# 4. 工具：SDK 会自动把 RunContextWrapper 注入进来                                                          
@function_tool                                                                                              
def get_user_plan(context: RunContextWrapper[UserContext]) -> str:                                          
    user = context.context                                                                                  
    return "pro" if user.is_pro_user else "free"                                                            
                                                                                                            
                                                                                                            
# 5. RunHooks：观察整个 Runner.run(...)                                                                     
class DebugRunHooks(RunHooksBase[UserContext, Agent[UserContext]]):                                         
    async def on_agent_start(self, context, agent):                                                         
        print("\n[HOOK] on_agent_start")                                                                    
        print("agent.name:", agent.name)                                                                    
        print("context type:", type(context).__name__)                                                      
        print("user:", context.context)                                                                     
                                                                                                            
    async def on_llm_start(self, context, agent, system_prompt, input_items):                               
        print("\n[HOOK] on_llm_start")                                                                      
        print("agent.name:", agent.name)                                                                    
        print("context type:", type(context).__name__)                                                      
        print("user:", context.context)                                                                     
        print("system_prompt:")                                                                             
        print(system_prompt)                                                                                
        print("input_items:")                                                                               
        for item in input_items:                                                                            
            print(item)                                                                                     
                                                                                                            
    async def on_llm_end(self, context, agent, response):                                                   
        print("\n[HOOK] on_llm_end")                                                                        
        print("agent.name:", agent.name)                                                                    
        print("response.output count:", len(response.output))                                               
        print("usage:", response.usage)                                                                     
                                                                                                            
        for i, output_item in enumerate(response.output, start=1):                                          
            print(f"output[{i}].type:", getattr(output_item, "type", None))                                 
            print(output_item)                                                                              
                                                                                                            
    async def on_tool_start(self, context, agent, tool):                                                    
        print("\n[HOOK] on_tool_start")                                                                     
        print("agent.name:", agent.name)                                                                    
        print("tool.name:", tool.name)                                                                      
        print("context type:", type(context).__name__)                                                      
        print("user:", context.context)                                                                     
                                                                                                            
        # 如果 context 是 ToolContext，通常还能看到 tool_call_id 等                                         
        print("tool_call_id:", getattr(context, "tool_call_id", None))                                      
        print("tool_name:", getattr(context, "tool_name", None))                                            
        print("tool_arguments:", getattr(context, "tool_arguments", None))                                  
                                                                                                            
    async def on_tool_end(self, context, agent, tool, result):                                              
        print("\n[HOOK] on_tool_end")                                                                       
        print("agent.name:", agent.name)                                                                    
        print("tool.name:", tool.name)                                                                      
        print("tool result:", result)                                                                       
        print("tool_call_id:", getattr(context, "tool_call_id", None))
                                                                                                            
    async def on_handoff(self, context, from_agent, to_agent):                                              
        print("\n[HOOK] on_handoff")                                                                        
        print("from:", from_agent.name)                                                                     
        print("to:", to_agent.name)                                                                         
                                                                                                            
    async def on_agent_end(self, context, agent, output):                                                   
        print("\n[HOOK] on_agent_end")                                                                      
        print("agent.name:", agent.name)                                                                    
        print("final output:", output)                                                                      
        print("total usage:", context.usage)                                                                
                                                                                                            
                                                                                                            
# 6. Agent                                                                                                  
agent = Agent[UserContext](                                                                                 
    name="User Assistant",                                                                                  
    model=model,                                                                                            
    instructions=dynamic_instructions,                                                                      
    tools=[get_user_plan],    
                                                                             
)                                                                                                           
                                                                                                            
                                                                                                            
# 7. 运行                                                                                                   
ctx = UserContext(                                                                                          
    name="Ada",                                                                                             
    uid="u_001",                                                                                            
    is_pro_user=True,                                                                                       
)                                                                                                           
                                                                                                            
result = await Runner.run(                                                                                  
    agent,                                                                                                  
    "我的会员等级是什么？",                                                                                 
    context=ctx,                                                                                            
    hooks=DebugRunHooks(),
    run_config=RunConfig(tracing_disabled=True),                                                                                       
)                                                                                                           
                                                                                                            
print("\n=== FINAL ===")                                                                                    
print(result.final_output)    


[HOOK] on_agent_start
agent.name: User Assistant
context type: AgentHookContext
user: UserContext(name='Ada', uid='u_001', is_pro_user=True)

[HOOK] on_llm_start
agent.name: User Assistant
context type: RunContextWrapper
user: UserContext(name='Ada', uid='u_001', is_pro_user=True)
system_prompt:
                                                                                             
You are helping the current user.                                                                           

Known user profile:                                                                                         
- name: Ada                                                                                         
- uid: u_001                                                                                           
- is_pro_user: True                                                                           

Rules:                                                                                  

In [ ]:
##强制选择工具  deepseek智能选择“auto模式”
from dataclasses import dataclass
from agents import Agent, Runner, RunConfig, function_tool, ModelSettings, RunContextWrapper, OpenAIChatCompletionsModel

from openai import AsyncOpenAI                                                                                                                                                      
                                                                                                                                
client = AsyncOpenAI(                                                                                                          
    api_key="sk-839ab4b91b25494d910a04a8812d8cf6",                                                                             
    base_url="https://api.deepseek.com",                                                                                       
)                                                                                                                              
                                                                                                                                
model = OpenAIChatCompletionsModel(                                                                                            
    model="deepseek-v4-pro",                                                                                                   
    openai_client=client,                                                                                                      
)                                                                                                                              
                  


@dataclass
class UserContext:
    name: str
    id: str
    city: str


@function_tool
def get_weather(ctx: RunContextWrapper[UserContext]) -> str:
    """Returns weather info for the specified city."""
    return f"The weather in {ctx.context.city} is sunny"

ctx = UserContext(
    name="ztz",
    id="test01",
    city='guangzhou'
)


agent = Agent(
    name="Weather Agent",
    model=model,
    instructions="Retrieve weather details.",
    tools=[get_weather],
    # model_settings=ModelSettings(tool_choice="auto")
)

result = await Runner.run(                                                                                  
    agent,                                                                                                  
    "今天天气如何？",   
    context=ctx,                                                                              
    run_config=RunConfig(tracing_disabled=True),  
                                                                                         
)                                                                                                           
                                                                                                            
print("\n=== FINAL ===")                                                                                    
# print(result.final_output)  
for item in result.new_items: 
    print(item)
    print("\n"*3) 


=== FINAL ===
ReasoningItem(agent=Agent(name='Weather Agent', handoff_description=None, tools=[FunctionTool(name='get_weather', description='Returns weather info for the specified city.', params_json_schema={'properties': {}, 'title': 'get_weather_args', 'type': 'object', 'additionalProperties': False, 'required': []}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x7f650111c450>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None)], mcp_servers=[], mcp_config={}, instructions='Retrieve weather details.', prompt=None, handoffs=[], model=<agents.models.openai_chatcompletions.OpenAIChatCompletionsModel object at 0x7f6501109050>, model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, paralle

In [2]:
# 工具使用行为  stop相关
##强制选择工具  deepseek智能选择“auto模式”
from dataclasses import dataclass
from agents import Agent, Runner, RunConfig, function_tool, RunContextWrapper, OpenAIChatCompletionsModel
from agents import StopAtTools
from openai import AsyncOpenAI                                                                                                                                                      
                                                                                                                                
client = AsyncOpenAI(                                                                                                          
    api_key="sk-839ab4b91b25494d910a04a8812d8cf6",                                                                             
    base_url="https://api.deepseek.com",                                                                                       
)                                                                                                                              
                                                                                                                                
model = OpenAIChatCompletionsModel(                                                                                            
    model="deepseek-v4-pro",                                                                                                   
    openai_client=client,                                                                                                      
)                                                                                                                              
                  


@dataclass
class UserContext:
    name: str
    id: str
    city: str


@function_tool
def get_weather(ctx: RunContextWrapper[UserContext]) -> str:
    """Returns weather info for the specified city."""
    return f"The weather in {ctx.context.city} is sunny"

ctx = UserContext(
    name="ztz",
    id="test01",
    city='guangzhou'
)


agent = Agent(
    name="Weather Agent",
    model=model,
    instructions="Retrieve weather details.",
    tools=[get_weather],
    # tool_use_behavior="stop_on_first_tool"
    tool_use_behavior=StopAtTools(stop_at_tool_names=["get_weather"])
)

result = await Runner.run(                                                                                  
    agent,                                                                                                  
    "今天天气如何？",   
    context=ctx,                                                                              
    run_config=RunConfig(tracing_disabled=True),  
                                                                                         
)                                                                                                           
                                                                                                            
print("\n=== FINAL ===")                                                                                    
# print(result.final_output)  
for item in result.new_items: 
    print(item)
    print("\n"*3) 



=== FINAL ===
ReasoningItem(agent=Agent(name='Weather Agent', handoff_description=None, tools=[FunctionTool(name='get_weather', description='Returns weather info for the specified city.', params_json_schema={'properties': {}, 'title': 'get_weather_args', 'type': 'object', 'additionalProperties': False, 'required': []}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x7f1416440450>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None)], mcp_servers=[], mcp_config={}, instructions='Retrieve weather details.', prompt=None, handoffs=[], model=<agents.models.openai_chatcompletions.OpenAIChatCompletionsModel object at 0x7f141644ba10>, model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, paralle

In [8]:
# 工具使用行为  自定义停止
from dataclasses import dataclass
from agents import Agent, Runner, RunConfig, function_tool, RunContextWrapper, OpenAIChatCompletionsModel
from agents.agent import FunctionToolResult, ToolsToFinalOutputResult
from openai import AsyncOpenAI       
from typing import List, Any                                                                                                                                               
                                                                                                                                
client = AsyncOpenAI(                                                                                                          
    api_key="sk-839ab4b91b25494d910a04a8812d8cf6",                                                                             
    base_url="https://api.deepseek.com",                                                                                       
)                                                                                                                              
                                                                                                                                
model = OpenAIChatCompletionsModel(                                                                                            
    model="deepseek-v4-pro",                                                                                                   
    openai_client=client,                                                                                                      
)                                                                                                                              
                  


@dataclass
class UserContext:
    name: str
    id: str
    city: str


@function_tool
def get_weather(ctx: RunContextWrapper[UserContext]) -> str:
    """Returns weather info for the specified city."""
    return f"The weather in {ctx.context.city} is sunny"


def custom_tool_handler(
    context: RunContextWrapper[UserContext],
    tool_results: List[FunctionToolResult],
)->ToolsToFinalOutputResult:
    for result in tool_results:
        if result.output and "sunny" in result.output:
            print(f"[DEBUG]  custom_tool_handler: {result.output}")
            return ToolsToFinalOutputResult(
                is_final_output=True,
                final_output=f"[FIX]: {result.output}"
            )
    return ToolsToFinalOutputResult(
        is_final_output=False,
        final_output=None
    )

ctx = UserContext(
    name="ztz",
    id="test01",
    city='guangzhou'
)


agent = Agent(
    name="Weather Agent",
    model=model,
    instructions="Retrieve weather details.",
    tools=[get_weather],
    # tool_use_behavior="stop_on_first_tool"
    # tool_use_behavior=custom_tool_handler
)

result = await Runner.run(                                                                                  
    agent,                                                                                                  
    "今天天气如何？",   
    context=ctx,                                                                              
    run_config=RunConfig(tracing_disabled=True),  
                                                                                         
)                                                                                                           
                                                                                                            
print("\n=== FINAL ===")                                                                                    
# print(result.final_output)  
for item in result.new_items: 
    print(item)
    print("\n"*3) 



=== FINAL ===
ReasoningItem(agent=Agent(name='Weather Agent', handoff_description=None, tools=[FunctionTool(name='get_weather', description='Returns weather info for the specified city.', params_json_schema={'properties': {}, 'title': 'get_weather_args', 'type': 'object', 'additionalProperties': False, 'required': []}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x7f14164d9150>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None)], mcp_servers=[], mcp_config={}, instructions='Retrieve weather details.', prompt=None, handoffs=[], model=<agents.models.openai_chatcompletions.OpenAIChatCompletionsModel object at 0x7f141648ed50>, model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, paralle